In [ ]:
# Empty for "ruff" extension

# Feature model time benchmark

This notebook focuses on the time required to compute fingerprints by different feature_models.

Todo:

- Test feature model time to fingerprint with different dataset sizes: 100 rows, qqp, something larger?
- Estimate how time grows with data, comparing them with a line graph
- Maybe measure memory used?


In [ ]:
import sys
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import time

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.fingerprint import FingerprintParams, create_fingerprints
from src.datasets import read_atn_sampled

%load_ext autoreload
%autoreload 2

In [ ]:
spark = get_spark()

MODELS = ["embeddings", "shingles (words)", "shingles (chars)", "bag_of_words"]
SHINGLE_WORDS_CONFIG = FingerprintParams(shingle_size=3, over_words=True)
SHINGLE_CHARS_CONFIG = FingerprintParams(shingle_size=3, over_words=False)
FINGERPRINT_SIZE = 64 # TODO: would it be interesting to also measure time when this changes?
BIG_SAMPLE_SIZES = [1000, 10000, 50000, 100000, 735849]
SMALL_SAMPLE_SIZES = [100, 500, 1000]

COL_IN = "_c0"
COL_OUT = "fingerprint1"

df = read_atn_sampled()

def benchmark(models: list[str], samples: list[int]) -> list[dict]:
    times = []
    for sample_size in samples:
        if sample_size != -1:
            df_sample = df.limit(sample_size)
        else:
            df_sample = df
            sample_size = df_sample.count()
        df_sample.count()
        for model in models:
            if model == "shingles (words)":
                start_time = time.perf_counter()
                df_temp = create_fingerprints(df=df_sample, model="shingles", input_col=COL_IN, output_col=COL_OUT, fingerprint_size=FINGERPRINT_SIZE, other=SHINGLE_WORDS_CONFIG)
                df_temp.count()
                finish_time = time.perf_counter()
                times.append({
                    "model": model,
                    "time": (finish_time - start_time) * 1000,
                    "fingerprint_size": FINGERPRINT_SIZE,
                    "sample_size": sample_size,
                })
                df_temp.unpersist()
            elif model == "shingles (chars)":
                start_time = time.perf_counter()
                df_temp = create_fingerprints(df=df_sample, model="shingles", input_col=COL_IN, output_col=COL_OUT, fingerprint_size=FINGERPRINT_SIZE, other=SHINGLE_CHARS_CONFIG)
                df_temp.count()
                finish_time = time.perf_counter()
                times.append({
                    "model": model,
                    "time": (finish_time - start_time) * 1000,
                    "fingerprint_size": FINGERPRINT_SIZE,
                    "sample_size": sample_size,
                })
                df_temp.unpersist()
            else:
                start_time = time.perf_counter()
                df_temp = create_fingerprints(df=df_sample, model=model, input_col=COL_IN, output_col=COL_OUT, fingerprint_size=FINGERPRINT_SIZE, other=None)
                df_temp.count()
                finish_time = time.perf_counter()
                times.append({
                    "model": model,
                    "time": (finish_time - start_time) * 1000,
                    "fingerprint_size": FINGERPRINT_SIZE,
                    "sample_size": sample_size,
                })
                df_temp.unpersist()
    return times

In [ ]:
df.persist()
big_times = benchmark(MODELS, BIG_SAMPLE_SIZES)
# small_times = benchmark(MODELS, SMALL_SAMPLE_SIZES)
df.unpersist()

In [ ]:
plot_df = pd.DataFrame(small_times)
plot_df["time"] = plot_df["time"].round(2)
plot_pivot = plot_df.pivot_table(index="sample_size", columns="model", values="time")
plt.figure(figsize=(10,6))
plot = sns.lineplot(
    data=plot_pivot,
    palette="deep"
)
plt.title("Time performance of feature models")
plt.xlabel("Size of sample")
plt.ylabel("Time (ms)")
plt.xlim(SMALL_SAMPLE_SIZES[0])
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
plot_df = pd.DataFrame(big_times)
plot_df["time"] = plot_df["time"].round(2)
plot_pivot = plot_df.pivot_table(index="sample_size", columns="model", values="time")
plt.figure(figsize=(10,6))
plot = sns.lineplot(
    data=plot_pivot,
    palette="deep"
)
# plt.title("Time performance of feature models (without tf-idf)")
plt.title("Time performance of feature models")
plt.xlabel("Size of sample")
plt.ylabel("Time (ms)")
plt.legend(title="Legend", loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

In [ ]:
spark.stop()